# $K$-armed bandit problem

### ***Summer School - June, 29 2026***

A statistician walks into a casino and is faced with several slot machines. Some of the machines pay out frequently when played, while others pay out much less often.  She has a certain number of plays available and aims to maximise her winnings. What are the best strategies? 


There is $K \in \mathbb{N}^\star$ slot machines in the casino. The learner has $T\in \mathbb{N}^\star$ moves to play and aims to maximise her winnings.

For $t = 1,\dots, T$

$\qquad \bullet$ Pick a slot machine $I_t \in \{1,\dots, K\}$

$\qquad \bullet$ Receive reward $g_t$ with $g_t \, | \, I_t = k \sim \nu_k$

We first simulate a completly random strategy (linear regret). 

In [ ]:
# import libraries
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython import display
random.seed(0)
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100

#  Choice of slot machine number
K = 3

# Generation of Bernoulli distribution parameters 
# p = np.random.uniform(0,1,K)

# Or manual definition of the distribution parameters
p = np.array([0.5,0.2,0.6])

# Horizon
T = 150

# Simulation of the gains (full-information setting)
# Matrix of size K x T that will contain the winnings from each machine 
G_fullinfo = np.random.binomial(1, p, size=(T, 3)).T

In [ ]:
def play_random(G_fullinfo):
    """
    G_fullinfo : Matrix K x T containing all the gains (full information settings)
    """
    # Vector of size T that will contain the numbers of the machines played 
    K, T = G_fullinfo.shape
    It = -1*np.ones(T)
    G = np.zeros([K,T])
    for t in range(T):
        # The learner chooses the slot machine at random. 
        it = math.floor(random.uniform(0,K))
        # it is stored in the vector It
        It[t] = it 
        # She received the reward gt : il est tiré selon la loi de Bernoulli de paramètre p[it]
        # For each slot machine
        for k in range(K):
            # if k is played, then the cumulative reward associated with machine k is updated
            if k == it:
                G[k,t] = G[k,t-1] + G_fullinfo[k,t]
            # if k is not played, then the cumulative reward associated with machine k stays the same
            else:
                G[k,t] = G[k,t-1] 
    return It, G

It_rd, G_rd = play_random(G_fullinfo)

## I. Explore then commit

The learner explores in a deterministic manner during $K \times T_{\mathrm{explore}}$ moves : she picks each slot machine $T_{\mathrm{explore}}$ times (*Explore* stage).
Then, she estimates $p_k$ for each $k$ (empirical mean).
The rest of the time, it will play the machine with the highest empirical mean, i.e. the one that earned it the most during the exploration phase (*Commit* stage).

In [ ]:
# Explore then commit 

def play_etc(G_fullinfo, prop_explore = 0.2):
    
    """
    G_fullinfo : Matrix K x T containing all the gains (full information settings)
    prop_explore : proportion of time dedicated to exploration 
    """
    
    # Vector of size T that will contain the numbers of the machines played 
    K, T = G_fullinfo.shape
    # Matrix of size K x T  that will contain the winnings from each machine 
    G_etc = np.zeros([K, T])

    # Vector of size T that will contain the numbers of the machines played 
    It_etc  = -1*np.ones(T)

    # Vector of size K that will contain the number of times each machine has been played 
    N = np.zeros([K])
    
    # T_explore : number of time each slot machine is played
    T_explore = math.floor(prop_explore*T/(K))

    # Explore then commit algorithm: 
    for t in range(T):
        # explore
        if t<(K*T_explore):
            it = math.floor(t/T_explore)

        #commit 
        else:
            it = np.argmax(G_etc[:,K*T_explore-1])
        It_etc[t] = it 
        for k in range(K):
            if k == it:
                G_etc[k,t] = G_etc[k,t-1] + G_fullinfo[k,t]
                N[k] = N[k] + 1
            else:
                G_etc[k,t] = G_etc[k,t-1] 
    return It_etc, G_etc


It_etc, G_etc = play_etc(G_fullinfo)

Sensitivity to exploration proportion factor:

In [ ]:
n = 1000
T = 5000

palette = [
"#223F6A",
"#294F8E",
"#326DC0",
"#6F8F78",
"#098D40",
"#7FA833",
"#D5B813"
]



# theory claims best alpha is alpha=1 (for Bernouilli distributions)
props = [0.05,0.25,0.5,0.75]
res_etc = np.zeros([len(props), n, T])

# pour chaque simulation i 
for i in range(n):
    G_fullinfo = np.random.binomial(1, p, size=(T, 3)).T
    for p_e in range(len(props)):
        _, G_etc = play_etc(G_fullinfo, prop_explore = props[p_e])
        res_etc[p_e, i, :] = np.sum(G_etc, axis=0)

x = np.linspace(0, T, T)
y = np.mean(res_etc, axis=1)
sd = np.std(res_etc, axis=1)

fig, ax = plt.subplots()
for p_e in range(len(props)):
    ax.plot(x, y[p_e,:], label=props[p_e], color=palette[p_e])
    ax.fill_between(x, y[p_e,:] + sd[p_e,:], y[p_e,:] - sd[p_e,:], alpha = 0.2, color=palette[p_e])
ax.set_xlabel ('Iterations')
ax.set_ylabel ('Cumulative reward')
ax.legend()
plt.show()

y_best = np.arange(1, T+1, 1)*max(p)

fig, ax = plt.subplots()
for p_e in range(len(props)):
    ax.plot(x, y_best - y[p_e,:], label=props[p_e], color=palette[p_e])
ax.set_xlabel ('Iterations')
ax.set_ylabel ('Regret')
ax.legend()
plt.show()

## II. Upper confidence bound

The *Upper confidence bound* (UCB, Auer et al., 2002) algorithm lies on the principle of optimism in the face of uncertainty.

For each $k=1,\dots,K$, the learner approximates $p_k$ (empirical mean - we denote $\widehat{p}_{k,t}$ the estimation of $p_k$ at $t$) and builds a confidence interval $\alpha_{k,t}$ arround $p_k$ such that, with high probability,  $p_k \in [\widehat{p}_{k,t} - \alpha_{k,t} ,\widehat{p}_{k,t} + \alpha_{k,t} ]$.
This is where optimism comes in: she imagines that all the machines are as profitable as is very likely possible; in other words, she imagines that for all slot machines, $p_k = \widehat{p}_{k,t} + \alpha_{k,t}$ and therefore chooses
$I_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}}\{ \widehat{p}_{k,t} + \alpha_{k,t} \}$. 

The challenge now lies in choosing the $\alpha_{k,t}$. The theory (Hoeffding inequality) gives the explicit expression:
$$I_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}} \Bigg\{ \widehat{p}_{k,t} + \sqrt{\frac{2 \log t}{N_{k,t}}} \Bigg\} $$
with $N_{k,t}$ the number of times the statistician plays machine $k$ after move $t$.

In [ ]:
def play_ucb(G_fullinfo, alpha = 1):
    """
    G_fullinfo : Matrix K x T containing all the gains (full information settings)
    alpha : exploration constant (theroy claims a=1 for Bernoulli distributions) 
    """
    K, T = G_fullinfo.shape

    G_ucb = np.zeros([K, T])
    It_ucb = -1*np.ones(T)
    ucb = np.zeros([K, T])
    N = np.zeros([K,T])

    for t in range(T):
        if t<K:
            it = t
        else:
            it = np.argmax(ucb[:,t-1])
        It_ucb[t] = it 
        for k in range(K):
            if k == it:
                G_ucb[k,t] = G_ucb[k,t-1] + G_fullinfo[k,t]
                if t == 1:
                    N[k,t] = 1*(k == it)
                else:
                    N[k,t] = N[k,t-1] + 1
            else:
                G_ucb[k,t] = G_ucb[k,t-1] 
                N[k,t] = N[k,t-1] 
            ucb[k,t] = G_ucb[k,t]/max(N[k, t],1) + alpha*math.sqrt(2*math.log(t+1)/max(N[k,t],1))
    return It_ucb, G_ucb, N

It_ucb, G_ucb, N = play_ucb(G_fullinfo)

def PlotUCB(t): 
    ylm = -0.7
    ylM = 2
    plt.cla()   
    plt.clf()
    plt.ylim((ylm,ylM))
    plt.xlim((-0.4,K-0.6))
    plt.axhline(y=0, color = 'red', linestyle = 'dotted')
    plt.axhline(y=1, color = 'red', linestyle = 'dotted')

    it = It_ucb[t]
    for k in range(K):
        if k == it:
            c = "#01A9B1"
        else:
            c = '#22509D'
        plt.axvline(x= k , 
                ymin= (G_ucb[k,t]/max(N[k, t],1) - (math.sqrt(2*math.log(max(t+1,2))/max(N[k,t],1)))-ylm )/(ylM-ylm),
                ymax= (G_ucb[k,t]/max(N[k, t],1) + (math.sqrt(2*math.log(max(t+1,2))/max(N[k,t],1))) -ylm  )/(ylM-ylm) ,
                color=c, 
                marker = 'x', markersize=7)
        plt.plot(k,(G_ucb[k,t]/max(N[k, t],1)),c,marker='o') 
        plt.plot(k,p[k],'k',marker='*', markersize=7) 
    plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    bottom=False,      # ticks along the bottom edge are off
    top=False,         # ticks along the top edge are off
    labelbottom=False)


f, ax1 = plt.subplots(1,1)
anim_created = animation.FuncAnimation(f,PlotUCB, frames=T, interval=300)
video = anim_created.to_jshtml()
html = display.HTML(video)
display.display(html) 
plt.close()


Sensitivity to exploration factor $\alpha$: 
$$I_{t+1} \in \underset{k \in \{1,\dots,K\}}{\mathrm{argmax}} \Bigg\{ \widehat{p}_{k,t} + \alpha\sqrt{\frac{2 \log t}{N_{k,t}}} \Bigg\} $$


In [ ]:
n = 500
T = 1000

# theory claims best alpha is alpha=1 (for Bernouilli distributions)
alphas = [0.001,0.01,0.1,1,10,100,1000]
res_ucb = np.zeros([len(alphas), n, T])

for i in range(n):
    G_fullinfo = np.random.binomial(1, p, size=(T, 3)).T
    
    for a in range(len(alphas)):
        _, G_ucb, _ = play_ucb(G_fullinfo, alpha = alphas[a])
        res_ucb[a, i, :] = np.sum(G_ucb, axis=0)


In [ ]:
x = np.linspace(0, T, T)
y = np.mean(res_ucb, axis=1)
sd = np.std(res_ucb, axis=1)

fig, ax = plt.subplots()
for a in range(len(alphas)):
    ax.plot(x, y[a,:], label=alphas[a], color=palette[a])
    ax.fill_between(x, y[a,:] + sd[a,:], y[a,:] - sd[a,:], alpha = 0.2, color=palette[a])
ax.set_xlabel ('Iterations')
ax.set_ylabel ('Cumulative reward')
ax.legend()
plt.show()

y_best = np.arange(1, T+1, 1)*max(p)

fig, ax = plt.subplots()
for a in range(len(alphas)):
    ax.plot(x, y_best - y[a,:], label=alphas[a], color=palette[a])
ax.set_xlabel ('Iterations')
ax.set_ylabel ('Regret')
ax.legend()
plt.show()

In the previous animation, each segment corresponds to the confidence interval $[\widehat{p}_{k,t} - \alpha_{k,t} ,\widehat{p}_{k,t} + \alpha_{k,t} ]$ for each slot machine $k = 1,\dots,K$, the black stars correspond to the true values $p_k$ and the circles to the estimated values $\widehat{p}_{k,t}$. The machine being played (the one with the highest confidence interval) lights up in light blue, as in the first animations.  

## III. Thompson Sampling

The Thompson sampling algorithm (Thompson, 1933; Chapelle and Li, 2011; Scott, 2010) is a Bayesian-sampling-based strategy.

For each $k=1,\dots,K$, the learner gives a prior distributions for $k$-machine rewawrds: a Beta distribution with parameters ($a_k$=1,$b_k$=1). 
For $t=1,\dots,T$, she will simulates the machine rewards by randomly drawing according to the prior distributions and then proceeds as if the simulated rewards were real: she chooses the machine with the highest ones. Then, based on the actual observed reward, she updates the distribution a posteriori: for the chosen machine $k$, $a_k$= $a_k$+1 if the reward is $1$; $b_k=b_k+1$ otherwise.

In [ ]:
# Thompson sampling
from scipy.stats import beta

def play_ts(G_fullinfo):
    
    """
    G_fullinfo : Matrix K x T containing all the gains (full information settings)
    """
    
    # Vector of size T that will contain the numbers of the machines played 
    K, T = G_fullinfo.shape
    # Matrix of size K x T  that will contain the winnings from each machine 
    G_ts = np.zeros([K, T])

    # Vector of size T that will contain the numbers of the machines played 
    It_ts = -1*np.ones(T)

    # Vector of size K that will contain the number of times each machine has been played 
    N = np.zeros([K])

    # A and B (Beta distribution parameters)
    A = np.ones([K])
    B = np.ones([K])
    g_sim = -1*np.ones([K])

    # Store parameters 
    A_ts = np.zeros([K, T])
    B_ts = np.zeros([K, T])


    # Explore then commit algorithm: 
    for t in range(T):
        A_ts[:,t] = A
        B_ts[:,t] = B 
        # simulate rewards
        g_sim = np.random.beta(A,B)

        it = np.argmax(g_sim)
        It_ts[t] = it 
        gt = G_fullinfo[it,t]

        if gt == 1:
            A[it] += 1
        else:
            B[it] += 1


        for k in range(K):
            if k == it:
                G_ts[k,t] = G_ts[k,t-1] + gt
                N[k] = N[k] + 1
            else:
                G_ts[k,t] = G_ts[k,t-1] 
    return It_ts, G_ts, A_ts, B_ts

It_ts, G_ts, A_ts, B_ts = play_ts(G_fullinfo)

def PlotTS(t): 
    ylm = 0
    ylM = 8
    plt.cla()   
    plt.clf()
    plt.ylim((ylm,ylM))
    plt.xlim((0,1))
    it = It_ts[t]
    ln = ['solid','dotted','dashed','dashdot']
    for k in range(K):
        if k == it:
            c = "#01A9B1"
        else:
            c = '#22509D'
        a = A_ts[k, t] 
        b = B_ts[k, t]
        x = np.linspace(0, 1, 1000)
        plt.plot(x, beta(a,b).pdf(x), color=c, linestyle= ln[k%len(ln)])
    plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    labelbottom=False)


f, ax1 = plt.subplots(1,1)
anim_created = animation.FuncAnimation(f,PlotTS, frames=T, interval=300)
video = anim_created.to_jshtml()
html = display.HTML(video)
display.display(html) 
plt.close()


In the previous animation, for each of the slot machines $k = 1,\dots,K$, with different line styles, the prior Beta distribution is plotted. The distribution of the machine being played (the one with the highest confidence interval) lights up in light blue. 

## IV. Comparison of strategies (Random, Explore then commit, Upper confidence bound, Thompson sampling)

In [ ]:
palette = {
    "Random": "#223F6A",
    "ETC": "#D5B813",
    "UCB": "#326DC0",
    "TS": "#098D40"
}

n = 5000

algos = {
    "Random": lambda G: play_random(G)[:2],
    "ETC":    lambda G: play_etc(G)[:2],
    "UCB":    lambda G: play_ucb(G)[:2],
    "TS":     lambda G: play_ts(G)[:2],
}

results = {name: np.zeros((n, T)) for name in algos}

# Simulations
for i in range(n):
    G_fullinfo = np.random.binomial(1, p, size=(T, K)).T

    for name, algo in algos.items():
        _, G = algo(G_fullinfo)
        results[name][i] = np.sum(G, axis=0)

x = np.arange(T)
means = {k: v.mean(axis=0) for k, v in results.items()}
stds  = {k: v.std(axis=0)  for k, v in results.items()}

fig, ax = plt.subplots()

for name in algos:
    ax.plot(x, means[name], label=name, color=palette[name])
    ax.fill_between(
        x,
        means[name] - stds[name],
        means[name] + stds[name],
        color=palette[name],
        alpha=0.2
    )

ax.set_xlabel("Iterations")
ax.set_ylabel("Cumulative reward")
ax.legend()
plt.show()

y_best = np.arange(1, T+1, 1)*max(p)
regret = {k: y_best - v.mean(axis=0) for k, v in results.items()}

fig, ax = plt.subplots()

for name in algos:
    ax.plot(x, regret[name], label=name, color=palette[name])

ax.set_xlabel("Iterations")
ax.set_ylabel("Regret")
ax.legend()
plt.show()